# A Neural Net From Scratch — and Why Backprop Is Just the Chain Rule

We'll build a 2-layer MLP in pure NumPy, hand-derive the gradients, and verify against PyTorch's autograd.

Network: `x → Linear(d, h) → ReLU → Linear(h, 1) → Sigmoid → BCE`

In [ ]:
import numpy as np
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Tiny toy dataset: 2 features, 100 examples, learnable boundary.
N, D, H = 100, 2, 8
X = np.random.randn(N, D).astype(np.float32)
y = ((X[:, 0] + X[:, 1] > 0).astype(np.float32))[:, None]

## The math

Forward:
$$ z_1 = X W_1 + b_1,\quad a_1 = \text{ReLU}(z_1),\quad z_2 = a_1 W_2 + b_2,\quad \hat y = \sigma(z_2) $$
$$ L = -\tfrac1N \sum [\, y \log \hat y + (1-y)\log(1-\hat y)\,] $$

Backward (chain rule, working from output back):
$$ \frac{\partial L}{\partial z_2} = \hat y - y $$
$$ \frac{\partial L}{\partial W_2} = a_1^\top \frac{\partial L}{\partial z_2},\quad
   \frac{\partial L}{\partial b_2} = \sum \frac{\partial L}{\partial z_2} $$
$$ \frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} W_2^\top $$
$$ \frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \cdot \mathbb{1}[z_1 > 0] $$
$$ \frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial z_1},\quad
   \frac{\partial L}{\partial b_1} = \sum \frac{\partial L}{\partial z_1} $$

Notice the elegant collapse: $\partial L / \partial z_2 = \hat y - y$ comes from cancelling the sigmoid's derivative against the BCE loss. That's why frameworks compose them as `BCEWithLogitsLoss`.

In [ ]:
# Init params
W1 = np.random.randn(D, H).astype(np.float32) * 0.1
b1 = np.zeros((1, H), dtype=np.float32)
W2 = np.random.randn(H, 1).astype(np.float32) * 0.1
b2 = np.zeros((1, 1), dtype=np.float32)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def forward_backward(X, y, W1, b1, W2, b2):
    # Forward
    z1 = X @ W1 + b1
    a1 = np.maximum(0, z1)
    z2 = a1 @ W2 + b2
    yhat = sigmoid(z2)
    loss = -np.mean(y * np.log(yhat + 1e-9) + (1 - y) * np.log(1 - yhat + 1e-9))

    # Backward
    N = X.shape[0]
    dz2 = (yhat - y) / N            # collapsed BCE+sigmoid grad
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)
    da1 = dz2 @ W2.T
    dz1 = da1 * (z1 > 0)            # ReLU derivative
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)
    return loss, (dW1, db1, dW2, db2)

# Train
lr = 0.5
for step in range(2001):
    loss, (dW1, db1, dW2, db2) = forward_backward(X, y, W1, b1, W2, b2)
    W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2
    if step % 500 == 0:
        print(f"step {step:4d}  loss {loss:.4f}")

z1 = np.maximum(0, X @ W1 + b1)
yhat = sigmoid(z1 @ W2 + b2)
print(f"\ntrain accuracy: {((yhat > 0.5) == y).mean():.3f}")

## Sanity check: same gradients from PyTorch autograd

We initialize a PyTorch model with the same weights, run one forward-backward, and confirm the gradients match the hand-derived ones.

In [ ]:
# Fresh init for comparison
np.random.seed(SEED); torch.manual_seed(SEED)
W1_np = np.random.randn(D, H).astype(np.float32) * 0.1
b1_np = np.zeros((1, H), dtype=np.float32)
W2_np = np.random.randn(H, 1).astype(np.float32) * 0.1
b2_np = np.zeros((1, 1), dtype=np.float32)

model = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, 1))
with torch.no_grad():
    model[0].weight.copy_(torch.from_numpy(W1_np.T))   # nn.Linear stores W as (out, in)
    model[0].bias.copy_(torch.from_numpy(b1_np.flatten()))
    model[2].weight.copy_(torch.from_numpy(W2_np.T))
    model[2].bias.copy_(torch.from_numpy(b2_np.flatten()))

Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
logits = model(Xt)
loss_t = nn.BCEWithLogitsLoss()(logits, yt)
loss_t.backward()

loss_np, (dW1, db1, dW2, db2) = forward_backward(X, y, W1_np, b1_np, W2_np, b2_np)

print(f"loss  | numpy: {loss_np:.6f}  | torch: {loss_t.item():.6f}")
print(f"dW1 max diff: {np.max(np.abs(dW1.T - model[0].weight.grad.numpy())):.2e}")
print(f"dW2 max diff: {np.max(np.abs(dW2.T - model[2].weight.grad.numpy())):.2e}")
print("\nGradients match to floating-point precision -> our derivation is correct.")

## Takeaways

- **Backprop is not magic** — it's bookkeeping for the chain rule on a DAG.
- **Sigmoid + BCE collapse** to a clean `yhat - y` gradient. Always use `BCEWithLogitsLoss` (or `cross_entropy` on logits) for numerical stability.
- **PyTorch / TF autograd** save you from writing this by hand — but knowing it pays off when debugging custom layers, gradient explosions, or numerical issues.